In [0]:
Raw CSV
   ↓
Ingest
   ↓
Transform
   ↓
Load
   ↓
Analytics-ready data

In [0]:
Extract
  ↓
Read raw CSV
  ↓
Transform
  ↓
Clean + calculate
  ↓
Load
  ↓
Delta table

In [0]:
                RAW DATA
              sales.csv
                  │
                  ▼
              EXTRACT
                  │
                  ▼
             PYSPARK
                  │
        ┌─────────┴─────────┐
        ▼                   ▼
     CLEAN               TRANSFORM
        │                   │
        │            Calculate revenue
        │            Standardize data
        │            Handle nulls
        └─────────┬─────────┘
                  ▼
              LOAD
                  │
                  ▼
            DELTA TABLE
                  │
                  ▼
            SQL ANALYSIS

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("RetailSalesETL").getOrCreate()

df_raw = spark.read.csv(
    "/Workspace/Users/sauichyansalai@gmail.com/sales.csv",
    header=True,
    inferSchema=True
)

df_raw.show()

In [0]:
df_raw.printSchema()

In [0]:
df_raw.show(10)

In [0]:
from pyspark.sql.functions import col, coalesce, lit

df_clean = df_raw.withColumn(
    "quantity",
    coalesce(col("quantity"), lit(1))
)

In [0]:
from pyspark.sql.functions import initcap

df_clean = df_clean.withColumn(
    "product",
    initcap(col("product"))
)

In [0]:
df_transformed = df_clean.withColumn(
    "total_amount",
    col("quantity") * col("price")
)

In [0]:
df_final = df_transformed.select(
    "order_id",
    "order_date",
    "customer_name",
    "product",
    "category",
    "quantity",
    "price",
    "total_amount"
)

In [0]:
df_final.show()

In [0]:
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_sales")

Python / PySpark
       ↓
Transform
       ↓
Delta
       ↓
retail_sales

In [0]:
%sql
SELECT *
FROM retail_sales;

In [0]:
%sql
SELECT
    SUM(total_amount) AS total_revenue
FROM retail_sales;

Revenue By Catagory

In [0]:
%sql
SELECT
    category,
    SUM(total_amount) AS revenue
FROM retail_sales
GROUP BY category
ORDER BY revenue DESC;

In [0]:
%sql
select product, sum(total_amount) as revenue
from retail_sales
group by product
order by revenue desc

In [0]:
retail-sales-etl-pipeline/
│
├── data/
│   └── sales.csv
│
├── notebooks/
│   └── retail_sales_etl.py
│
├── sql/
│   └── analysis_queries.sql
│
├── README.md
│
└── architecture.png